# AoC 2024 Day 12 — Garden Groups

**Spark — connected components by min-label propagation**

Puzzle: <https://adventofcode.com/2024/day/12>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A map of garden plots, one letter per plot. Plots of the same letter that touch **horizontally or vertically** form a region — diagonals do not connect, and the same letter can form several separate regions.

Each region has an **area** (how many plots) and a **perimeter** (how many of its plots' four sides do not touch another plot of the same region — including sides facing the outside of the map, and sides facing a different region enclosed within it).

- **Part 1** — the price of a region is `area × perimeter`. Sum the price of every region.

## The approach

Two things fall out of the **same edge list**, which is why building it is the only parsing step.

Explode the grid to `(node, r, c, ch)` and self-join it to its right and down neighbours where the plant matches. Symmetrise that (union it with itself, swapped) and you have `(u, v)` for every same-plant adjacency.

**Perimeter needs no second pass.** Every plot has four sides; a side is fence exactly when it does *not* face a same-plant neighbour. So `perimeter = 4 - degree`, and degree is a `groupBy(u).count()` on the edge list already built. Cells with no neighbours never appear in the edges at all, hence the `left` join and `coalesce(deg, 0)` — a lone plot correctly scores 4.

**Regions are connected components**, and that is the interesting part. The loop is min-label propagation: each node starts labelled with its own id, and each round takes the smallest label one edge away. Done naively that spreads one hop per round, so a snaking region 200 cells long needs 200 rounds of joins — ruinous.

The fix is two extra moves per round:

- **hook the parent too** — proposals are emitted for `pu` as well as `u`, so a whole group of nodes sharing a label is dragged down together rather than one at a time
- **pointer doubling** — after hooking, join the label table to itself (`p == gnode`) and take `least(p, gp)`, halving every chain in one step

Together those turn *O(region diameter)* rounds into *O(log n)*. The `140×140` input converges in a handful of rounds instead of a few hundred.

**And it is still the slowest day in the repo: ~6.4 s against ~46 ms for the plain Python BFS flood fill.** That is not a defect in the Spark code — it is what iterative graph work costs when every round is a shuffle. 19600 cells fit in L2 cache; a BFS touches each once and never leaves the CPU. The Spark version pays a full job launch, several joins and a `count()` barrier *per round*. The relational framing is worth understanding precisely because it is the one that keeps working at 140 million cells — but at 140×140 it loses, and loses badly.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day12

spark = get_spark('aoc-2024-day12')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = 'RRRRIICCFF\nRRRRIICCCF\nVVRRRCCFFF\nVVRCCCJFFF\nVVVVCJJCFE\nVVIVCCJJEE\nVVIIICJJEE\nMIIIIIJJEE\nMIIISIJEEE\nMMMISSJEEE\n'

print('part 1:', day12.part1(spark, EXAMPLE), '(expected 1930)')

### One edge list, both answers

The degree column below is doing double duty — it is the perimeter *and* it is the graph the components loop walks. Compare the region table against the prices the puzzle lists for this example (`R` at 12 × 18 = 216, and so on).

In [ ]:
from pyspark.sql import functions as F

# Small example only -- the CC loop is a shuffle per round.
grid = day12.cells(spark, EXAMPLE).cache()
adjacency = day12.edges(grid).cache()
print('cells:', grid.count(), ' directed edges:', adjacency.count())

# Perimeter is 4 - degree, straight off the edge list. No second pass.
degree = adjacency.groupBy(F.col('u').alias('node')).agg(F.count('*').alias('deg'))
grid.join(degree, "node", "left").select(
    'r',
    'c',
    'ch',
    F.coalesce('deg', F.lit(0)).alias('deg'),
    (F.lit(4) - F.coalesce('deg', F.lit(0))).alias('perimeter'),
).orderBy('r', 'c').show(8)

# Components: label = smallest node id in the region.
parent = day12.components(grid, adjacency)
regions = (
    grid.join(parent, "node")
    .join(degree, "node", "left")
    .groupBy("p")
    .agg(
        F.first("ch").alias("plant"),
        F.count("*").alias("area"),
        F.sum(F.lit(4) - F.coalesce("deg", F.lit(0))).alias("perimeter"),
    )
    .withColumn("price", F.col("area") * F.col("perimeter"))
)
regions.orderBy("p").show()
print("total:", regions.agg(F.sum("price").alias("t")).collect()[0]["t"])

adjacency.unpersist()
grid.unpersist()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 12)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day12.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day12 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- `node` is a **row-major id**, `r * width + c`, with `width = longest line + 1`. The `+ 1` is the guard: without it the last cell of row *r* and the first cell of row *r+1* can collide into the same id, silently welding two regions together. It is cast to `long` because a big grid overflows `int`.
- `split(line, '')` emits a **trailing empty string**, filtered out in `cells()`. Same trap as day 4.
- Edges are built **right and down only**, then unioned with the swap. Enumerating all four directions directly would produce every edge twice and double the degree.
- The loop's termination test is a `count()` of changed labels — an **action**, so every round is a real job. That is deliberate: there is no way to know a fixpoint has been reached without looking. It is also why `localCheckpoint()` sits inside the loop; the lineage would otherwise grow one full round of joins per iteration.
- Labels converge to the **smallest node id** in the region, which is stable and deterministic, but it is an arbitrary identifier — do not read meaning into its value.
- The perimeter formula assumes **grid cells only**. Off-grid neighbours are perimeter by construction, since a cell outside the grid never produced an edge, so no explicit boundary handling appears anywhere.
- Ragged input (lines of differing length) would break the `width` assumption. AoC grids are rectangular; the code takes the max length defensively but the geometry still assumes a rectangle.